### 1. 형태소 분석기를 이용하여 품사가 명사인 경우 해당 단어를 추출

In [1]:
# !cd ~/data && unzip synopsis.zip -d ~/work/weat/data/

In [2]:
import os

In [2]:
data_dir = os.path.join(os.getenv("HOME"), "work/weat/data")
file_name = os.path.join(data_dir, "synopsis.txt")

with open(file_name, 'r') as file:
    for i in range(20):
        print(file.readline(), end='')

NameError: name 'os' is not defined

In [4]:
# !sudo apt update
# !sudo apt install openjdk-17-jdk -y
# !echo 'export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))' >> ~/.bashrc
# !source ~/.bashrc

# !pip install konlpy


In [1]:
from konlpy.tag import Okt

okt = Okt()
tokenized = []

with open(file_name, 'r') as file:
    while True:
        line = file.readline()
        if not line : break
        # stem=True: 활용된 단어를 기본형으로 변환 ㅡ 어간 추출
        # 즉 동/형을 원형으로 복원 (ex: 먹었다 -> 먹다)
        words = okt.pos(line, stem=True, norm=True)
        res = []
        for w in words:
            if w[1] in ['Noun']:  # "Adjective", "Verb" 등을 포함할 수 도 있다
                res.append(w[0])  # 명사일 때 만 tokenized 에 저장하게 된다
        tokenized.append(res)

print('~')

NameError: name 'file_name' is not defined

In [ ]:
print(len(tokenized))

In [ ]:
tokenized[:10]

### 2. 추출된 결과로 embedding model 만들기

In [ ]:
from gensim.models import Word2Vec

In [ ]:
# tokenized 에 담긴 데이터를 가지고 나만의 Word2Vec 을 생성
# 텍스트 데이터를 가지고 단어 임베딩을 학습하는 과정
model = Word2Vec(tokenized, vector_size=100, window=5, min_count=3, sg=0)
model.wv.most_similar(positive=['영화'])

- vector_size = 임베딩 차원
- window = context 범위
- min_count = 최소 등장 횟수 -> 노이즈 제거, 학습 안정화
- sg : 모델 종류

### 3. TF-IDF 로 해당 데이터를 가장 잘 표현하는 단어 셋 만들기

In [ ]:
import os
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from konlpy.tag import Okt

### TF-IDF를 적용했을 때의 문제점

- **단어의 중요도는 반영하지만 의미는 반영 X**

 - TF-IDF를 이용해도 중복된 단어를 잘 제거하면 여전히 유용한 방식

In [ ]:
def read_token(file_name):
    okt = Okt()
    result = []
    with open(data_dir + '/' + file_name, 'r') as fread:
        print(file_name, '파일을 읽고 있습니다.')
        while True:
            line = fread.readline()
            if not line: break
            tokenlist = okt.pos(line, stem=True, norm=True)
            for word in tokenlist:
                if word[1] in ["Noun"]:#, "Adjective", "Verb"]:
                    result.append((word[0]))
    return ' '.join(result)


In [ ]:
genre_txt = ['synopsis_SF.txt', 'synopsis_family.txt', 'synopsis_show.txt', 'synopsis_horror.txt', 'synopsis_etc.txt',
             'synopsis_documentary.txt', 'synopsis_drama.txt', 'synopsis_romance.txt', 'synopsis_musical.txt',
             'synopsis_mystery.txt', 'synopsis_crime.txt', 'synopsis_historical.txt', 'synopsis_western.txt',
             'synopsis_adult.txt', 'synopsis_thriller.txt', 'synopsis_animation.txt', 'synopsis_action.txt',
             'synopsis_adventure.txt', 'synopsis_war.txt', 'synopsis_comedy.txt', 'synopsis_fantasy.txt']

genre_name = ['SF', '가족', '공연', '공포(호러)', '기타', '다큐멘터리', '드라마', '멜로로맨스', '뮤지컬', '미스터리', '범죄', '사극', '서부극(웨스턴)',
         '성인물(에로)', '스릴러', '애니메이션', '액션', '어드벤처', '전쟁', '코미디', '판타지']
print("슝~")

In [ ]:
genre = []
for file_name in genre_txt:
    genre.append(read_token(file_name))

In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(genre)

print(X.shape)

In [ ]:
m1 = X[0].tocoo() # art 를 TF-IDF로 표현한 sparse matrix를 가져온다
m2 = X[1].tocoo() # gen 을 TF-IDF로 표현한 sparse matrix를 가져온다

w1 = [[i,j] for i, j in zip(m1.col, m1.data)]
w2 = [[i,j] for i, j in zip(m2.col, m2.data)]

w1.sort(key=lambda x: x[1], reverse=True) # art를 구성하는 단어들을 TF-IDF가 높은 순으로 정렬
w2.sort(key=lambda x: x[1], reverse=True) # gen을 구성하는 단어들을 TF-IDF가 높은 순으로 정렬


print('예술영화를 대표하는 단어들:')
for i in range(100):
    print(vectorizer.get_feature_names_out()[w1[i][0]], end=', ')

print('\n')

print('일반영화를 대표하는 단어들:')
for i in range(100):
    print(vectorizer.get_feature_names_out()[w2[i][0]], end=', ')

n = 15
w1_, w2_ =[], []
for i in range(100):
    w1_.append(vectorizer.get_feature_names_out()[w1[i][0]])
    w2_.append(vectorizer.get_feature_names_out()[w2[i][0]])

# w1에만 있고 w2에는 없넌, 예술 영화를 잘 대표하는 단어를 15개 추출한다
target_art, target_gen =[] ,[]
for i in range(100):
    if (w1_[i] not in w2_) and (w1_[i] in model.wv) :
        target_art.append(w1_[i])
    if len(target_art) ==n:
        break

# w2에만 있고, w1에는 없는, 일반 영화를 잘 대표하는 단어를 15개 추출한다
for i in range(100):
    if (w2_[i] not in w1_) and (w2_[i] in model.wv):
        target_gen.append(w2_[i])
    if len(target_gen) ==n:
        break

## **유사한 거 중복 제거**

In [ ]:
def is_similar(word, selected, model,threshold=0.8):
    for w in selected:
        if model.wv.similarity(word,w) > threshold:
            return True
    return False

In [ ]:
m = [X[i].tocoo() for i in range(X.shape[0])]

w = [[[i, j] for i, j in zip(mm.col, mm.data)] for mm in m]

for i in range(len(w)):
    w[i].sort(key=lambda x: x[1], reverse=True)
attributes = []
for i in range(len(w)):
    print(genre_name[i], end=': ')
    attr = []
    j = 0

    # 상위 15개 단어를 뽑을 때 까지
    while (len(attr) < 15):

        word = vectorizer.get_feature_names_out()[w[i][j][0]]

        if word in model.wv:
            if not is_similar(word, attr, model):
                attr.append(word)
                print(word, end=', ')
        
        j+=1
    attributes.append(attr)
    print()

### 4. embedding model 과 단어 셋으로 WEAT score 구해보기

- target_X = art
- target_Y = gen
- attribute_A = 'Drama'
- attribute_B = 'Action'

$$
\frac{
\mathrm{mean}_{x \in X} s(x, A, B)
-
\mathrm{mean}_{y \in Y} s(y, A, B)
}{
\mathrm{std}_{w \in X \cup Y} s(w, A, B)
}
$$

- 두 target 집단 (X,Y)이 attribute(A,B)에 얼마나 다르게 연관되는지를 표준화해서 측정하는 값
- mean_x : x집단이 A,B 중 어디에 더 가까운지 평균
- mean_y : y집단이 A,B 중 어디에 더 가까운지 평균
- mean_x - mean_y : x와 y가 얼마나 다르게 행동하는지
- /std_dev : 두 집단 차이를 표준화

In [ ]:

def cos_sim(i,j):
    return np.dot(i, j.T)/(np.linalg.norm(i)*np.linalg.norm(j))

def s(w, A, B):
    c_a = cos_sim(w, A)
    c_b = cos_sim(w, B)
    mean_A = np.mean(c_a, axis=-1)
    mean_B = np.mean(c_b, axis=-1)
    return mean_A - mean_B

def weat_score(X, Y, A, B):

    s_X = s(X, A, B)
    s_Y = s(Y, A, B)

    mean_X = np.mean(s_X)
    mean_Y = np.mean(s_Y)

    std_dev = np.std(np.concatenate([s_X, s_Y], axis=0))

    return  (mean_X-mean_Y)/std_dev

In [ ]:
matrix = [[0 for _ in range(len(genre_name))] for _ in range(len(genre_name))]

In [ ]:
X = np.array([model.wv[word] for word in target_art])
Y = np.array([model.wv[word] for word in target_gen])

for i in range(len(genre_name)-1):
    for j in range(i+1, len(genre_name)):
        A = np.array([model.wv[word] for word in attributes[i]])
        B = np.array([model.wv[word] for word in attributes[j]])
        matrix[i][j] = weat_score(X,Y,A,B)
print('~')

In [ ]:
# for i in range(len(genre_name)-1):
#     for j in range(i+1, len(genre_name)):
#         print(genre_name[i], genre_name[j], matrix[i][j])

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
np.random.seed(0)
# 마이너스 부호

plt.figure(figsize=(15, 12))  # (가로, 세로) 크게 조절

ax = sns.heatmap(
    matrix,
    xticklabels=genre_name,
    yticklabels=genre_name,
    annot=True,
    cmap='RdYlGn_r'
)

plt.xticks(rotation=45)  # x축 글자 기울이기
plt.yticks(rotation=0)

plt.show()

### 회고

-행(ROW)기준 해석
- 미스터리 장르는 범죄(0.92) / 스릴러(0.83) 과 높은 유사도를 보이며, 이는 해당 장르들이 사건이 공통괸 특성을 공유하기 때문이다
- 다큐멘터리가 다른 장르들과 유사하지 않다고 나온다
- 멜로로멘스는 미스터리랑 거리가 가장 낮고(0.41), 애니메이션이랑 가장 거리가 가깝게(0.87) 나온다.
- SF가 현실기반 다큐멘러리와 가장 양의 값을 가지는게 우리가 생각하는 거랑 다르게 나온다


- 몇몇 결과가 직관과 다르게 나왔다. 이는 실제 의미 관계가 왜곡 되었다라고 생각한다. 해결하기 위해서 유사도가 높은 단어는 제외했는데, 생각보다 효과가 없었던 것 같다

- NLP 는 이미지랑 다르게 처리하는 과정이 익숙하지 않다. 퍼실님은 NLP가 CV보다 재밌다고 하셔서 어디서 재미를 느끼셨는지 이해해 보고 싶어 포인트를 찾으려고 했지만, 실패했다.